In [1]:
from datascience import Table
from datetime import datetime

def convert_date(date_str):
    date_str = date_str.strip().strip("'")
    return datetime.strptime(date_str, '%Y-%m-%d/%H:%M')

def calc_drift_rate(file_path):
    possibles = Table.read_table(file_path)
    
    high_freq = possibles.column(2)
    low_freq = possibles.column(3)
    freq_change = high_freq - low_freq
    
    start_time = possibles.apply(convert_date, "Start time")
    end_time = possibles.apply(convert_date, "End time")
    difference = end_time - start_time
    duration_seconds = [diff.total_seconds() for diff in difference]
    
    possibles = possibles.with_column("Change in Frequency", freq_change).with_column("Duration (seconds)", duration_seconds)

    drift_rate = possibles.column("Change in Frequency") / possibles.column("Duration (seconds)")
    possibles = possibles.with_column("Drift Rate (Hz/s)", drift_rate)

    drift_rate_mhz = drift_rate/1000000
    possibles = possibles.with_column("Drift Rate (MHz/s)", drift_rate_mhz)
    possibles.show()

    return possibles

In [3]:
possibles_dfdt = calc_drift_rate("possibles.csv")

Start time,End time,High frequency (Hz),Low frequency (Hz),Change in Frequency,Duration (seconds),Drift Rate (Hz/s),Drift Rate (MHz/s)
2023-1-1/01:30,2023-1-1/04:00,1e+07,50000,9.95e+06,9000,1105.56,0.00110556
2023-1-3/06:25,2023-1-3/09:00,2e+07,70000,1.993e+07,9300,2143.01,0.00214301
2023-1-4/03:14,2023-1-4/03:20,1e+07,600000,9.4e+06,360,26111.1,0.0261111
2023-1-4/05:53,2023-1-4/06:15,2e+07,300000,1.97e+07,1320,14924.2,0.0149242
2023-1-4/03:14,2023-1-4/03:29,1e+07,600000,9.4e+06,900,10444.4,0.0104444
2023-1-4/09:19,2023-1-4/09:24,2e+07,600000,1.94e+07,300,64666.7,0.0646667
2023-1-4/12:34,2023-1-4/12:45,1e+07,1e+06,9e+06,660,13636.4,0.0136364
2023-1-4/14:31,2023-1-4/14:34,2e+07,500000,1.95e+07,180,108333,0.108333
2023-1-5/17:58,2023-1-5/18:50,2e+07,600000,1.94e+07,3120,6217.95,0.00621795
2023-1-6/05:43,2023-1-6/05:55,1e+07,400000,9.6e+06,720,13333.3,0.0133333


In [5]:
from pytplot import get_data
from matplotlib import pyplot as plt
%matplotlib tk
from matplotlib.colors import LogNorm

In [17]:
low_freq = possibles_dfdt.column("Low frequency (Hz)") / 1000000
high_freq = possibles_dfdt.column("High frequency (Hz)") / 1000000

plt.scatter(low_freq, high_freq)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Low frequency (MHz)')
plt.ylabel('High frequency (MHz)')

plt.show()

In [13]:
duration = possibles_dfdt.column("Duration (seconds)")
high_freq = possibles_dfdt.column("High frequency (Hz)") / 1000000

plt.scatter(duration, high_freq)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Duration (s)')
plt.ylabel('High frequency (MHz)')

plt.show()

In [15]:
duration = possibles_dfdt.column("Duration (seconds)")
freq_diff = possibles_dfdt.column("Change in Frequency") / 1000000

plt.scatter(duration, freq_diff)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Duration (s)')
plt.ylabel('Freq Diff (MHz)')

plt.show()